# Rozdział 7 — Gdy struktura sygnału zmienia się w czasie

Wykonywalne doświadczenia E7.1–E7.15 do samodzielnego tekstu książki.
Źródła: kanoniczny dokument CPSB M6 oraz notebook M6 v.3.
Generatory kontrolne zachowują parametry źródeł; wszystkie dane są syntetyczne,
bez zapisu pacjenta i bez pobierania plików.
Wyjaśnienia konieczne do interpretacji zawiera rozdział, nie tylko notebook.

Zależności podano w requirements_ch07.txt. Uruchomienie z katalogu pakietu
lub notebooks/, ewentualnie przez python scripts/run_ch07.py.
Kod wspólny jest w src/bspsbook/. Figury są osobnymi PNG/SVG, bez osadzania
obrazów w komórkach. Poniższe odwołania wymagają zachowania struktury pakietu.


In [1]:
from pathlib import Path
import sys,json,platform,importlib.metadata
import numpy as np
import scipy,pywt,matplotlib
from scipy import signal
import matplotlib.pyplot as plt
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent] if (p/'src/bspsbook/time_frequency.py').is_file()),None)
if ROOT is None:raise RuntimeError('Uruchom z katalogu pakietu lub notebooks/.')
sys.path.insert(0,str(ROOT/'src'))
from bspsbook.plotting import configure_plots,save_figure
from bspsbook.time_frequency import stft_map,manual_psd,cwt_map,power_db,connected_width,band_power
from bspsbook.wavelets import components,denoise,metrics
from bspsbook.signals import (gaussian_envelope,make_tf_puzzle,make_stft_intro_signal,
    make_parameter_signal,make_window_test_signal,make_eeg_like_case_s4,
    make_eeg_emg_case_s5,make_two_band_signal,make_synthetic_ecg,make_ecg_wavelet_case,
    make_mismatched_case)
configure_plots(chapter=7);FIG=ROOT/'figures/generated';CHECKS=[];RESULTS={};PASSPORTS={}
def check(name,condition):
    if not bool(condition):raise AssertionError(name)
    CHECKS.append(name);print('OK:',name)
def rmse(a,b):return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
def tfplot(ax,t,f,p,title,kind='STFT',reference=None,limits=(-55,0),ylim=(0,90)):
    m=ax.pcolormesh(t,f,power_db(p,reference),shading='auto',cmap='viridis',
                    vmin=limits[0],vmax=limits[1],rasterized=True)
    ax.set(xlabel='Czas [s]',ylabel='Częstotliwość [Hz]' if kind=='STFT' else 'Pseudoczęstotliwość [Hz]',
           title=title,ylim=ylim)
    label=('PSD [dB re 1 (j.u.)²/Hz]' if reference==1 else
           ('PSD [dB re max tej mapy]' if kind=='STFT' else '|CWT|² [dB re max tej mapy]'))
    fig=ax.get_figure();fig.colorbar(m,ax=ax,label=label)
    return m
ENV={'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,
     'matplotlib':matplotlib.__version__,'PyWavelets':pywt.__version__,
     'nbformat':importlib.metadata.version('nbformat'),'nbclient':importlib.metadata.version('nbclient'),
     'ipykernel':importlib.metadata.version('ipykernel'),
     'rng':'numpy.random.default_rng / PCG64',
     'seeds':[202606,6301,6302,6303,6403,6404,6502,6606,6701,6706,6803]}
print(json.dumps(ENV,ensure_ascii=False,indent=2))


{
  "python": "3.12.14",
  "numpy": "2.3.5",
  "scipy": "1.17.0",
  "matplotlib": "3.10.8",
  "PyWavelets": "1.8.0",
  "nbformat": "5.10.4",
  "nbclient": "0.10.2",
  "ipykernel": "6.30.1",
  "rng": "numpy.random.default_rng / PCG64",
  "seeds": [
    202606,
    6301,
    6302,
    6303,
    6403,
    6404,
    6502,
    6606,
    6701,
    6706,
    6803
  ]
}


## E7.1 — to samo widmo, inna chronologia; rys. 7.1

fs200 Hz, trzy bloki po2 s, seed202606, szumSD0.03.
KolejnośćA:12 Hz, paczka35 Hz, chirp18–45 Hz; B:chirp,12 Hz,paczka.
Każdy blok otrzymuje własny szum przed permutacją. B jest przesunięciem kołowym
o400 próbek. Kontrola niecyklicznej permutacji oddziela ten szczególny przypadek.
Welch: Hann okresowy, segment400, brak overlap, detrend constant, density.

![Rys. 7.1](../figures/generated/fig_07_01_chronology.png)


In [2]:
t,xa,xb,oa,ob,blocks,nblock=make_tf_puzzle();fs=200.
check('E7.1: B jest przesunięciem kołowym całego A',np.array_equal(xb,np.roll(xa,nblock)))
Fa=np.fft.rfft(xa);Fb=np.fft.rfft(xb)
check('E7.1: moduły FFT zgodne mimo chronologii',np.allclose(abs(Fa),abs(Fb),atol=1e-12))
welch_kw=dict(fs=fs,window=signal.get_window('hann',400),nperseg=400,noverlap=0,detrend='constant',scaling='density')
fw,pa=signal.welch(xa,**welch_kw);_,pb=signal.welch(xb,**welch_kw)
check('E7.1: Welch uśrednia ten sam zbiór ramek',np.allclose(pa,pb,atol=1e-14))
xc=np.concatenate([blocks[oa[1]],blocks[oa[0]],blocks[oa[2]]])
relative_noncyclic=float(np.linalg.norm(abs(np.fft.rfft(xc))-abs(Fa))/np.linalg.norm(abs(Fa)))
check('E7.1: niecykliczna permutacja nie zachowuje dowolnie modułu',relative_noncyclic>.01)
phase_factor=np.exp(-2j*np.pi*np.arange(len(Fa))*nblock/len(xa))
check('E7.1: pełna DFT przechowuje zmianę fazy',np.allclose(Fb,Fa*phase_factor,atol=1e-10))
fig,ax=plt.subplots(2,2,figsize=(10,6.6),layout='constrained')
for a,x,label,order in [(ax[0,0],xa,'A',oa),(ax[0,1],xb,'B',ob)]:
    a.plot(t,x,lw=.6);a.set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title=label+': '+' → '.join(['12 Hz','35 Hz','chirp'] if label=='A' else ['chirp','12 Hz','35 Hz']))
    for cut in [2,4]:a.axvline(cut,color='black',ls=':',lw=.8)
f=np.fft.rfftfreq(len(xa),1/fs)
for X,label,ls in [(Fa,'A','-'),(Fb,'B','--')]:
    p=abs(X)**2/len(xa)**2;p[1:-1]*=2
    ax[1,0].plot(f,power_db(p,1),ls,label=label)
for p,label,ls in [(pa,'A','-'),(pb,'B','--')]:ax[1,1].plot(fw,power_db(p,1),ls,label=label)
ax[1,0].set(xlim=(0,60),ylim=(-75,0),xlabel='Częstotliwość [Hz]',ylabel='Moc [dB re 1 (j.u.)²]',title='Pełny rekord: moduł² FFT / N²')
ax[1,1].set(xlim=(0,60),ylim=(-75,0),xlabel='Częstotliwość [Hz]',ylabel='PSD [dB re 1 (j.u.)²/Hz]',title='Welch: kolejność ramek nie jest wynikiem')
for a in ax[1]:a.legend()
RESULTS['E7.1']={'FFT_max_module_difference':float(np.max(abs(abs(Fa)-abs(Fb)))),
    'Welch_max_abs_difference':float(np.max(abs(pa-pb))),'noncyclic_relative_module_change':relative_noncyclic}
save_figure(fig,FIG,'fig_07_01_chronology')


OK: E7.1: B jest przesunięciem kołowym całego A
OK: E7.1: moduły FFT zgodne mimo chronologii
OK: E7.1: Welch uśrednia ten sam zbiór ramek
OK: E7.1: niecykliczna permutacja nie zachowuje dowolnie modułu
OK: E7.1: pełna DFT przechowuje zmianę fazy
Figura: figures/generated/fig_07_01_chronology.png (+ SVG)


## E7.2 — od lokalnej DFT do skalowanej STFT; rys. 7.2

Model M6.2: fs200 Hz, T6 s, chirp10–50 Hz a0.75 plus paczka35 Hz
a1.15, centrum1.7 s, sigma0.14 s. Hann okresowy, L128, H32, NFFT256,
bez detrendu, wyłącznie pełne ramki. Czas umownego środka: start+L//2.
ShortTimeFFT phase_shift=None zapewnia zgodność zespolonej DFT z ręczną ramką;
przy phase_shift=0 moduły są identyczne, a fazy mają inną konwencję.
PSD jednego boku: q|FFT(xw)|²/(fs*sum(w²)), q=2 we wnętrzu,1 naDC/Nyquist.

![Rys. 7.2](../figures/generated/fig_07_02_stft_construction.png)


In [3]:
t,x,xc,xp,fref,center=make_stft_intro_signal()
st=stft_map(x,nfft=256);manual,tm=manual_psd(x,200.,128,32,256)
check('E7.2: osie ręcznych i bibliotecznych ramek zgodne',np.allclose(st['time'],tm))
check('E7.2: zespolone współczynniki PSD zgodne',np.allclose(st['coeff'],manual,atol=1e-13))
check('E7.2: zgodny krok i siatka',np.allclose(np.diff(tm),.16) and np.isclose(st['f'][1],.78125))
frames=np.array([x[s:s+128] for s in st['starts']]);w=st['window']
integral=st['power'].sum(axis=0)*(200/256)
weighted=np.sum((frames*w)**2,axis=1)/np.sum(w*w)
check('E7.2: suma PSD razy df daje ważony drugi moment',np.allclose(integral,weighted,atol=1e-13))
odd=stft_map(x,nfft=257);om,ot=manual_psd(x,200,128,32,257)
check('E7.2: nieparzyste FFT ma prawidłowe podwojenie',np.allclose(odd['coeff'],om,atol=1e-13))
tn=np.arange(128)/200;xx=1.7*np.sin(2*np.pi*25*tn);raw=np.fft.rfft(xx*w)
check('E7.2: osobne skalowanie amplitudy tonu',np.isclose(2*abs(raw[16])/sum(w),1.7))
tc=np.arange(800)/200;cc=np.exp(2j*np.pi*25*tc)+.6*np.exp(-2j*np.pi*37.5*tc)
cs=stft_map(cc,nfft=128)
ratio=float(cs['power'][cs['f']<0].sum()/cs['power'][cs['f']>0].sum())
check('E7.2: zespolone DQ zachowuje obie półosie bez podwajania',np.isclose(ratio,.36,atol=1e-12))
fig,ax=plt.subplots(3,1,figsize=(9.4,8),layout='constrained')
ax[0].plot(t,x,lw=.7);ax[0].set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Sygnał i trzy wybrane ramki')
for time in [.8,1.7,4.8]:
    i=int(np.argmin(abs(st['time']-time)));actual=st['time'][i]
    ax[0].axvspan(st['starts'][i]/200,(st['starts'][i]+128)/200,alpha=.12)
    ax[1].plot(st['f'],st['power'][:,i],label=f't={actual:.2f} s')
ax[1].set(xlim=(0,65),xlabel='Częstotliwość [Hz]',ylabel='PSD [(j.u.)²/Hz]',title='b) Każda kolumna pochodzi z całego okna');ax[1].legend(ncol=3)
tfplot(ax[2],st['time'],st['f'],st['power'],'c) Te same lokalne obliczenia w szeregu czasowym',reference=1,limits=(-55,-3),ylim=(0,65))
PASSPORTS['E7.2']=st['passport']
RESULTS['E7.2']={'complex_max_difference':float(np.max(abs(st['coeff']-manual))),
    'Parseval_max_abs_error':float(np.max(abs(integral-weighted))),'complex_negative_positive_energy_ratio':ratio,
    'first_time_s':float(tm[0]),'last_time_s':float(tm[-1]),'frames':len(tm)}
save_figure(fig,FIG,'fig_07_02_stft_construction')


OK: E7.2: osie ręcznych i bibliotecznych ramek zgodne
OK: E7.2: zespolone współczynniki PSD zgodne
OK: E7.2: zgodny krok i siatka
OK: E7.2: suma PSD razy df daje ważony drugi moment
OK: E7.2: nieparzyste FFT ma prawidłowe podwojenie
OK: E7.2: osobne skalowanie amplitudy tonu
OK: E7.2: zespolone DQ zachowuje obie półosie bez podwajania
Figura: figures/generated/fig_07_02_stft_construction.png (+ SVG)


## E7.3 — długość okna zmienia rozdzielanie i rozmycie; rys. 7.3

Model M6.3, fs200 Hz,6 s, seed6301. Gauss sigma0.48 s w2.4 s:
0.85sin28 +0.75sin32. Krótka paczka w4.16 s sigma0.045:
1.30*(0.65sin62+0.45sin72+0.30sin82). Dodatkowa paczkaEMG w5.05 s
sigma0.20 oraz szumSD0.015. L64/128/256, H16, NFFT512, Hann okresowy.
Mapy mają wspólne bezwzględne odniesienie PSD=1 i zakres(-55,-3)dB.
Przekroje normalizowane do własnego maksimum pokazują wyłącznie kształt.
Szerokość połączonego maksimum mocy przy połowie wysokości, interpolacja liniowa.

![Rys. 7.3](../figures/generated/fig_07_03_window_length.png)


In [4]:
t,x,xclose,xshort,xemg,short_center=make_parameter_signal();window_results={}
fig=plt.figure(figsize=(10.8,8.8),layout='constrained');gs=fig.add_gridspec(2,6,hspace=.17);top_axes=[]
bottom=[fig.add_subplot(gs[1,:3]),fig.add_subplot(gs[1,3:])]
rows=[]
for i,L in enumerate([64,128,256]):
    result=stft_map(x,length=L,hop=16,nfft=512);window_results[L]=result
    a=fig.add_subplot(gs[0,i*2:(i+1)*2])
    top_axes.append(a)
    shared_mesh=a.pcolormesh(result['time'],result['f'],power_db(result['power'],1),shading='auto',vmin=-55,vmax=-3,cmap='viridis',rasterized=True)
    a.set(xlabel='Czas [s]',ylabel='Częstotliwość [Hz]' if i==0 else '',xlim=(0,6),ylim=(0,90),title=f'L={L}, T={L/200:.2f} s')
    j=int(np.argmin(abs(result['time']-2.4)));p=result['power'][:,j]
    bottom[0].plot(result['f'],p/p.max(),label=f'L={L}')
    trace=band_power(result,60,85);roi=(result['time']>=3.3)&(result['time']<=4.85)
    width=connected_width(result['time'][roi],trace[roi],.5)
    bottom[1].plot(result['time'],trace/trace[roi].max(),label=f'L={L}; FWHM={width["width"]:.3f} s')
    rows.append({'L':L,'window_s':L/200,'df_Hz':200/512,'H':16,'band_60_85_FWHM_s':width['width'],
                 'band_peak_s':width['peak'],'width_clipped':width['clipped']})
fig.colorbar(shared_mesh,ax=top_axes,label='PSD [dB re 1 (j.u.)²/Hz]')
bottom[0].set(xlim=(23,37),xlabel='Częstotliwość [Hz]',ylabel='PSD / maksimum [1]',title='Separacja 28 i 32 Hz')
bottom[1].set(xlim=(3.3,4.85),xlabel='Czas środka ramki [s]',ylabel='Moc pasma / maksimum [1]',title='Krótka paczka 60–85 Hz')
for a in bottom:a.legend(fontsize=7)
check('E7.3: wspólne H i df wszystkich wariantów',all(np.isclose(r['df_Hz'],200/512) and r['H']==16 for r in rows))
check('E7.3: długie okno poszerza odpowiedź krótkiej paczki',rows[-1]['band_60_85_FWHM_s']>rows[0]['band_60_85_FWHM_s'])
check('E7.3: mierzone szerokości nie są obcięte',all(not r['width_clipped'] for r in rows))
RESULTS['E7.3']=rows
save_figure(fig,FIG,'fig_07_03_window_length')


OK: E7.3: wspólne H i df wszystkich wariantów
OK: E7.3: długie okno poszerza odpowiedź krótkiej paczki
OK: E7.3: mierzone szerokości nie są obcięte
Figura: figures/generated/fig_07_03_window_length.png (+ SVG)


## E7.4 — okno, overlap i zero-padding są różnymi decyzjami; rys. 7.4

Hann, Hamming, Tukey alpha0.25 przyL128/H32/NFFT512, model silnej28 i słabej42 Hz
zM6.3, seed6302. Overlap0/50/75%: ten sam xshort+szumSD0.004, seed6303,
L128/NFFT512, wspólne bezwzględne wartości mocy pasma, bez normalizacji każdego maksimum.
NFFT128/512/2048: identyczna ramka w2.4 s i to samo oknoL128.

![Rys. 7.4](../figures/generated/fig_07_04_sampling_and_windows.png)


In [5]:
tw,xw=make_window_test_signal();fig,ax=plt.subplots(2,2,figsize=(10.3,7.5),layout='constrained')
window_rows=[]
for win,name in [('hann','Hann'),('hamming','Hamming'),(('tukey',.25),'Tukey α=0,25')]:
    w=signal.get_window(win,128);F=np.fft.rfft(w,8192);fw=np.fft.rfftfreq(8192,1/200)
    ax[0,0].plot(fw,20*np.log10(np.maximum(abs(F)/abs(F).max(),1e-12)),label=name)
    r=stft_map(xw,window=win);j=int(np.argmin(abs(r['time']-2.4)))
    ax[0,1].plot(r['f'],power_db(r['power'][:,j],1),label=name)
    window_rows.append({'window':name,'PSD_at42Hz':float(r['power'][np.argmin(abs(r['f']-42)),j])})
ax[0,0].set(xlim=(0,20),ylim=(-100,3),xlabel='Odsunięcie od centrum [Hz]',ylabel='Moduł względny [dB]',title='a) Widma okien tej samej długości')
ax[0,1].set(xlim=(20,50),ylim=(-85,0),xlabel='Częstotliwość [Hz]',ylabel='PSD [dB re 1 (j.u.)²/Hz]',title='b) Silna 28 Hz i słaba 42 Hz')
xo=xshort+.004*np.random.default_rng(6303).normal(size=len(xshort));overlap_rows=[]
for H in [128,64,32]:
    r=stft_map(xo,hop=H);trace=band_power(r,60,85)
    ax[1,0].plot(r['time'],trace,'o-',ms=3,label=f'H={H}; {100*(1-H/128):.0f}%')
    overlap_rows.append({'H':H,'frames':len(trace),'dt_s':H/200,'max_band_power':float(trace.max()),
                         'peak_s':float(r['time'][np.argmax(trace)])})
ax[1,0].axvline(4.16,color='black',ls=':',lw=.9)
ax[1,0].set(xlim=(3.3,4.9),xlabel='Czas środka ramki [s]',ylabel='Moc pasma [(j.u.)²]',title='c) Krok może ominąć maksimum paczki')
nfft_rows=[];all_grids=[]
for nf in [128,512,2048]:
    r=stft_map(xclose,nfft=nf);j=int(np.argmin(abs(r['time']-2.4)))
    ax[1,1].plot(r['f'],r['power'][:,j],'.-' if nf==128 else '-',label=f'NFFT={nf}')
    all_grids.append(r['power'][:,j])
    nfft_rows.append({'NFFT':nf,'df_Hz':200/nf,'window_s':.64})
check('E7.4: zero-padding zachowuje wspólne prążki PSD',np.allclose(all_grids[0],all_grids[1][::4],atol=1e-14) and np.allclose(all_grids[1],all_grids[2][::4],atol=1e-14))
check('E7.4: więcej overlap daje więcej kolumn',np.all(np.diff([r['frames'] for r in overlap_rows])>0))
ax[1,1].set(xlim=(23,37),xlabel='Częstotliwość [Hz]',ylabel='PSD [(j.u.)²/Hz]',title='d) Więcej punktów tego samego widma')
for a in ax.flat:a.legend(fontsize=7)
RESULTS['E7.4']={'windows':window_rows,'overlap':overlap_rows,'zero_padding':nfft_rows}
save_figure(fig,FIG,'fig_07_04_sampling_and_windows')


OK: E7.4: zero-padding zachowuje wspólne prążki PSD
OK: E7.4: więcej overlap daje więcej kolumn
Figura: figures/generated/fig_07_04_sampling_and_windows.png (+ SVG)


## E7.5 — ramki brzegowe zawierają założenie o otoczeniu; rys. 7.5

Ten sam sygnał M6.2; L128/H32/NFFT256/Hann. Pełne mapy: dopełnienie zeros lubedge.
Maska complete wynika z faktycznych indeksów wszystkich L próbek okna,
nie z niezerowych wag. Obie mapy i przekrój mają to samo odniesieniePSD=1.

![Rys. 7.5](../figures/generated/fig_07_05_stft_edges.png)


In [6]:
t,x,*_=make_stft_intro_signal();zero=stft_map(x,nfft=256,valid=False,padding='zeros')
edge=stft_map(x,nfft=256,valid=False,padding='edge');valid=stft_map(x,nfft=256)
diff=abs(zero['power']-edge['power']);mask=zero['complete']
check('E7.5: padding nie zmienia kompletnych ramek',np.max(diff[:,mask])<1e-14)
check('E7.5: padding zmienia ramki brzegowe',np.max(diff[:,~mask])>1e-3)
check('E7.5: valid pokrywa te same rzeczywiste ramki',np.allclose(valid['power'],zero['power'][:,mask]) and np.allclose(valid['time'],zero['time'][mask]))
fig,ax=plt.subplots(3,1,figsize=(9.4,8.3),layout='constrained')
for a,r,name in [(ax[0],zero,'a) Przedłużenie zerowe'),(ax[1],edge,'b) Powtórzenie wartości brzegowej')]:
    tfplot(a,r['time'],r['f'],r['power'],name,reference=1,limits=(-55,-3),ylim=(0,65))
    for v in [valid['time'][0],valid['time'][-1]]:a.axvline(v,color='white',ls=':',lw=1.2)
ax[2].plot(zero['time'],np.max(diff,axis=0))
ax[2].set(xlabel='Czas umownego środka ramki [s]',ylabel='Maks. |różnica PSD| [(j.u.)²/Hz]',title='c) Różnica jest ograniczona do ramek korzystających z paddingu')
RESULTS['E7.5']={'full_first_last_s':zero['time'][[0,-1]].tolist(),'valid_first_last_s':valid['time'][[0,-1]].tolist(),
    'full_frames':len(zero['time']),'valid_frames':len(valid['time']),
    'max_padding_difference_interior':float(np.max(diff[:,mask])),
    'max_padding_difference_border':float(np.max(diff[:,~mask]))}
PASSPORTS['E7.5']={'zero':zero['passport'],'edge':edge['passport']}
save_figure(fig,FIG,'fig_07_05_stft_edges')


OK: E7.5: padding nie zmienia kompletnych ramek
OK: E7.5: padding zmienia ramki brzegowe
OK: E7.5: valid pokrywa te same rzeczywiste ramki
Figura: figures/generated/fig_07_05_stft_edges.png (+ SVG)


## E7.6 — atomy CWT i przeliczenie skali; rys. 7.6

cmor1.5-1.0, nominalny nośnik[-8,8], skale3/6/12, fs200 Hz.
Atomy psi(tau)/sqrt(a), czas b+a*tau/fs, przesunięcie dlaa6:0.075 s.
Osobna kontrola osi2–80 Hz,64 skalegeometryczne; po zmianiefs200→250
te same Hz wymagają skal razy1.25. sampling_period w PyWavelets zmienia
etykiety częstotliwości, nie współczynniki przy niezmiennych próbkach i skalach.

![Rys. 7.6](../figures/generated/fig_07_06_cwt_atoms.png)


In [7]:
wave='cmor1.5-1.0';mother=pywt.ContinuousWavelet(wave);psi,tau=mother.wavefun(level=10)
fig,ax=plt.subplots(3,1,figsize=(9.3,6.8),sharex=True,layout='constrained')
for a,scale in zip(ax,[3.,6.,12.]):
    shift=.075 if scale==6 else 0.;times=shift+scale*tau/200
    atom=psi/np.sqrt(scale);a.plot(times,atom.real,label='Re ψ');a.plot(times,abs(atom),'--',label='|ψ|')
    a.set(ylabel='Wartość atomu [j.u.]',title=f'a={scale:g}; pseudoczęstotliwość={pywt.scale2frequency(wave,scale)*200:.2f} Hz; b={shift:g} s')
    a.legend(ncol=2,fontsize=7)
ax[-1].set_xlabel('Czas [s]');ax[-1].set_xlim(-.5,.6)
freq=np.geomspace(2,80,64);scales=pywt.central_frequency(wave)*200/freq
check('E7.6: skala i Hz przeliczają się w obie strony',np.allclose(pywt.scale2frequency(wave,scales)*200,freq))
check('E7.6: stałe Hz po zmianie fs wymagają nowych skal',np.allclose((pywt.central_frequency(wave)*250/freq)/scales,1.25))
xx=np.sin(np.arange(300)*.5);a,fa=pywt.cwt(xx,[4,8],wave,sampling_period=1/200)
b,fb=pywt.cwt(xx,[4,8],wave,sampling_period=1/250)
check('E7.6: sampling_period nie skaluje amplitud współczynników',np.array_equal(a,b) and np.allclose(fb/fa,1.25))
RESULTS['E7.6']={'central_frequency':pywt.central_frequency(wave),'scale_range_for2_80Hz':scales[[0,-1]].tolist(),
    'nominal_support':[mother.lower_bound,mother.upper_bound],'scale_ratio_fs250_to200':1.25}
save_figure(fig,FIG,'fig_07_06_cwt_atoms')


OK: E7.6: skala i Hz przeliczają się w obie strony
OK: E7.6: stałe Hz po zmianie fs wymagają nowych skal
OK: E7.6: sampling_period nie skaluje amplitud współczynników
Figura: figures/generated/fig_07_06_cwt_atoms.png (+ SVG)


## E7.7 — te same dane i chwile oceny STFT/CWT; rys. 7.7

Chirp8–70 Hz w6 s, fs200 Hz, szumSD0.025, seed6403.
STFT:Hann,L128,H16,NFFT512, valid. CWT:cmor1.5-1.0,180 częstotliwości5–80 Hz.
Grzbiet = maksimum mocy w paśmie5–80 Hz. Ocena obu reprezentacji na tych samych
środkach STFT z przedziału0.8–5.2 s; CWT pobrana w tych próbkach.
Oddzielne odniesienia do maksimum każdej mapy, oba zakresy[-45,0]dB.

![Rys. 7.7](../figures/generated/fig_07_07_chirp_comparison.png)


In [8]:
fs=200.;t=np.arange(1200)/fs;truth=8+62*t/6
x=signal.chirp(t,f0=8,f1=70,t1=6)+.025*np.random.default_rng(6403).normal(size=len(t))
st=stft_map(x,hop=16);cw=cwt_map(x,frequencies=np.geomspace(5,80,180))
band=(st['f']>=5)&(st['f']<=80);rs=st['f'][band][np.argmax(st['power'][band],axis=0)]
rc=cw['f'][np.argmax(cw['power'],axis=0)];indices=np.rint(st['time']*fs).astype(int)
valid=(st['time']>=.8)&(st['time']<=5.2);ref=truth[indices]
maes=float(np.mean(abs(rs[valid]-ref[valid])));maec=float(np.mean(abs(rc[indices][valid]-ref[valid])))
check('E7.7: oba grzbiety oceniane na tej samej siatce czasu',np.allclose(t[indices],st['time']) and valid.sum()>40)
check('E7.7: kontrolny chirp zachowany w obu mapach',maes<2 and maec<2)
fig,ax=plt.subplots(3,1,figsize=(9.5,8.3),layout='constrained')
tfplot(ax[0],st['time'],st['f'],st['power'],'a) STFT: stała długość okna',limits=(-45,0),ylim=(5,80))
tfplot(ax[1],t,cw['f'],cw['power'],'b) CWT: zmienny czas atomu',kind='CWT',limits=(-45,0),ylim=(5,80))
for a in ax[:2]:a.plot(t,truth,'w--',lw=1);a.set_xlim(0,6)
ax[2].plot(st['time'][valid],rs[valid]-ref[valid],label=f'STFT MAE={maes:.3f} Hz')
ax[2].plot(st['time'][valid],rc[indices][valid]-ref[valid],label=f'CWT MAE={maec:.3f} Hz')
ax[2].set(xlabel='Wspólne czasy oceny [s]',ylabel='Błąd grzbietu [Hz]',title='c) Kształt mapy i dokładność to różne wyniki');ax[2].legend()
RESULTS['E7.7']={'MAE_STFT_Hz':maes,'MAE_CWT_Hz':maec,'common_evaluation_count':int(valid.sum()),
    'first_last_evaluation_s':st['time'][valid][[0,-1]].tolist()}
PASSPORTS['E7.7']={'STFT':st['passport'],'CWT':cw['passport']}
save_figure(fig,FIG,'fig_07_07_chirp_comparison')


OK: E7.7: oba grzbiety oceniane na tej samej siatce czasu
OK: E7.7: kontrolny chirp zachowany w obu mapach
Figura: figures/generated/fig_07_07_chirp_comparison.png (+ SVG)


## E7.8 — paczka oscylacji i stan przejściowy; rys. 7.8

Dwa osobne syntetyczne modele zM6, fs200 Hz,T8 s.
Lewy:EEG13 Hz,centrum2.4 s,sigma0.45,a0.75; EMG-podobne45/55/68 Hz,
centrum5.3 s,sigma0.12,a0.42*(1,0.75,0.55),fazy0.2/1/2;
tło0.12sin3Hz(phi0.4)+0.05sin7Hz(phi1.1)+szumSD0.035,seed6502.
Prawy:EEG13Hz,centrum3 s,sigma0.45,a0.75; dodatni Gauss a1.15,5.35 s,
sigma0.035; tło0.20sin3Hz+0.10sin7Hz+szumSD0.06,seed6404.
Hann,L128,H16,NFFT512 i CWTcmor1.5-1.0,128 częstotliwości4–80 Hz.
Szerokości mocy przy połowie maksimum mierzone w jawnym otoczeniu danej paczki.
Nie są to dane półsyntetyczne: cały sygnał generowany lokalnie.

![Rys. 7.8](../figures/generated/fig_07_08_biomedical_maps.png)


In [9]:
te,xee,bg,eeg,emg,truth=make_eeg_emg_case_s5()
tt,xet,bgt,spindle,transient,*_=make_eeg_like_case_s4()
fig,ax=plt.subplots(3,2,figsize=(11,9.5),layout='constrained');width_rows=[]
for col,(time,x,title) in enumerate([(te,xee,'EEG- i EMG-podobne paczki'),(tt,xet,'Wrzeciono i stan przejściowy')]):
    ax[0,col].plot(time,x,lw=.6);ax[0,col].set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title=title)
    st=stft_map(x,hop=16);cw=cwt_map(x,frequencies=np.geomspace(4,80,128))
    tfplot(ax[1,col],st['time'],st['f'],st['power'],'STFT',limits=(-55,0),ylim=(4,80))
    tfplot(ax[2,col],time,cw['f'],cw['power'],'CWT',kind='CWT',limits=(-55,0),ylim=(4,80))
    if col==0:
        for label,r,axis in [('STFT',st,st['time']),('CWT',cw,time)]:
            for event,freq,center,radius in [('EEG',13.,2.4,1.3),('EMG',55.,5.3,.7)]:
                row=int(np.argmin(abs(r['f']-freq)));roi=abs(axis-center)<=radius
                measure=connected_width(axis[roi],r['power'][row,roi],.5)
                width_rows.append({'method':label,'event':event,'actual_frequency_Hz':float(r['f'][row]),'FWHM_s':measure['width'],'clipped':measure['clipped']})
        st_bio,cw_bio=st,cw
for a in ax.flat:a.set_xlim(0,8)
check('E7.8: szerokości lokalnych paczek są mierzalne',all(r['FWHM_s']>0 and not r['clipped'] for r in width_rows))
doubled_st=stft_map(2*xee,hop=16);doubled_cw=cwt_map(2*xee,frequencies=np.geomspace(4,80,128))
check('E7.8: podwojenie amplitudy mnoży obie moce przez4',np.allclose(doubled_st['power'],4*st_bio['power']) and np.allclose(doubled_cw['power'],4*cw_bio['power']))
check('E7.8: własne maksimum ukrywa zmianę poziomu',np.allclose(power_db(doubled_st['power'],None),power_db(st_bio['power'],None),atol=1e-10) and np.allclose(power_db(doubled_cw['power'],None),power_db(cw_bio['power'],None),atol=1e-10))
single55=.42*.75*gaussian_envelope(te,5.3,.12)*np.sin(2*np.pi*55*te+1.)
only_emg=cwt_map(emg,frequencies=np.geomspace(4,80,128));only55=cwt_map(single55,frequencies=np.geomspace(4,80,128))
row=int(np.argmin(abs(only55['f']-55)));roi=abs(te-5.3)<=.7
single_width=connected_width(te[roi],only55['power'][row,roi])['width']
mixed_width=connected_width(te[roi],only_emg['power'][row,roi])['width']
check('E7.8: wąska wyspa mieszaniny nie mierzy całej obwiedni',single_width>2*mixed_width)
RESULTS['E7.8']={'isolated_55Hz_CWT_FWHM_s':single_width,'noiseless_multitone_CWT_FWHM_s':mixed_width,
                 'generator_power_envelope_FWHM_s':float(2*np.sqrt(np.log(2))*.12),
                 'widths_at_selected_frequencies':width_rows,'synthetic_not_semisynthetic':True,
                 'power_ratio_2x_over_x_STFT':float(doubled_st['power'].max()/st_bio['power'].max()),
                 'power_ratio_2x_over_x_CWT':float(doubled_cw['power'].max()/cw_bio['power'].max())}
save_figure(fig,FIG,'fig_07_08_biomedical_maps')


OK: E7.8: szerokości lokalnych paczek są mierzalne
OK: E7.8: podwojenie amplitudy mnoży obie moce przez4
OK: E7.8: własne maksimum ukrywa zmianę poziomu
OK: E7.8: wąska wyspa mieszaniny nie mierzy całej obwiedni
Figura: figures/generated/fig_07_08_biomedical_maps.png (+ SVG)


## E7.9 — brzegi i parametr falki; rys. 7.9

Identyczna paczka12 Hz,sigma0.18 s,rekord6 s,fs200 Hz,centra3 oraz0.05 s.
Nośna ma fazę względem środka paczki, aby przesunięcie nie zmieniało jej kształtu.
CWT3–50 Hz,96 punktów,cmor1.5-1.0. Obie mapy mają wspólny ref=max obu mocy.
Granica ostrzegawcza =8*a/fs z nominalnego nośnika[-8,8]; nie standardowy COI.
Dolne przekroje: izolowane wrzeciono13 Hz,sigma0.45,centrum3 s oraz izolowany
Gauss sigma0.035 w5.35 s; cmor0.8-1.0,cmor1.5-1.0,cmor1.5-1.5.
Szerokość połączonego maksimum; bez łączenia oddzielnych wysp powyżej połowy.

![Rys. 7.9](../figures/generated/fig_07_09_cwt_edges_parameters.png)


In [10]:
t=np.arange(1200)/200;freq=np.geomspace(3,50,96);edge_maps=[]
for center in [3.,.05]:
    burst=gaussian_envelope(t,center,.18)*np.sin(2*np.pi*12*(t-center))
    edge_maps.append(cwt_map(burst,frequencies=freq))
shared=max(r['power'].max() for r in edge_maps)
fig,ax=plt.subplots(2,2,figsize=(10.8,7.6),layout='constrained')
for a,r,center in zip(ax[0],edge_maps,[3.,.05]):
    m=a.pcolormesh(t,r['f'],power_db(r['power'],shared),shading='auto',vmin=-60,vmax=0,cmap='viridis',rasterized=True)
    a.plot(r['guard'],r['f'],'w--',lw=1);a.plot(t[-1]-r['guard'],r['f'],'w--',lw=1)
    a.axvline(center,color='white',ls=':',lw=1)
    a.set(xlabel='Czas [s]',ylabel='Pseudoczęstotliwość [Hz]',title=f'Środek paczki {center:g} s',ylim=(3,50))
    fig.colorbar(m,ax=a,label='|CWT|² [dB re wspólne maksimum]')
guard=edge_maps[0]['guard']
check('E7.9: niskie częstotliwości mają szerszą strefę brzegu',guard[0]>guard[-1])
band=(freq>=9)&(freq<=15)
edge_ratio=float(edge_maps[1]['power'][band].max()/edge_maps[0]['power'][band].max())
check('E7.9: obcięta paczka różni się od centralnej',abs(edge_ratio-1)>.05)
t2=np.arange(1600)/200;sp=.75*gaussian_envelope(t2,3,.45)*np.sin(2*np.pi*13*t2)
trans=1.15*gaussian_envelope(t2,5.35,.035);freq2=np.geomspace(3,70,180);rows=[]
for wave in ['cmor0.8-1.0','cmor1.5-1.0','cmor1.5-1.5']:
    spm=cwt_map(sp,frequencies=freq2,wavelet=wave);trm=cwt_map(trans,frequencies=freq2,wavelet=wave)
    j=round(3*200);profile=spm['power'][:,j];roi=(freq2>=7)&(freq2<=20)
    fw=connected_width(freq2[roi],profile[roi])
    near=(freq2>=11)&(freq2<=15);trace=spm['power'][near].max(axis=0);tw=connected_width(t2,trace)
    trband=(freq2>=10)&(freq2<=60);trtrace=trm['power'][trband].max(axis=0);trw=connected_width(t2,trtrace)
    ax[1,0].plot(freq2,profile/profile.max(),label=wave)
    ax[1,1].plot(t2-5.35,trtrace/trtrace.max(),label=wave)
    rows.append({'wavelet':wave,'spindle_frequency_FWHM_Hz':fw['width'],'spindle_time_FWHM_s':tw['width'],'transient_FWHM_s':trw['width']})
ax[1,0].set(xlim=(7,20),xlabel='Pseudoczęstotliwość [Hz]',ylabel='Moc / maksimum [1]',title='Izolowane wrzeciono: szerokość w Hz')
ax[1,1].set(xlim=(-.25,.25),xlabel='Czas względem centrum [s]',ylabel='Moc / maksimum [1]',title='Izolowany stan przejściowy: szerokość w czasie')
for a in ax[1]:a.legend(fontsize=7)
RESULTS['E7.9']={'edge_to_center_power_ratio':edge_ratio,'Morlet_parameters':rows}
save_figure(fig,FIG,'fig_07_09_cwt_edges_parameters')


OK: E7.9: niskie częstotliwości mają szerszą strefę brzegu
OK: E7.9: obcięta paczka różni się od centralnej
Figura: figures/generated/fig_07_09_cwt_edges_parameters.png (+ SVG)


## E7.10 — DWT jako bank filtrów z decymacją; rys. 7.10

fs200 Hz,T4 s. Wolny4 Hz a0.9, paczka42 Hz a0.45 (2.15 s,sigma0.35),
Gauss a0.65(1.2 s,sigma0.025). db4, filtry8 współczynników, symmetric.
Współczynniki a1/d1 rysowane po indeksie, A1/D1 po rekonstrukcji na osi czasu.
Pasma gałęzi są przybliżone; pojedyncza gałąź nie daje gwarancji braku aliasingu.

![Rys. 7.10](../figures/generated/fig_07_10_dwt_filter_bank.png)


In [11]:
t,x,slow,fast,pulse=make_two_band_signal();wave=pywt.Wavelet('db4')
a1,d1=pywt.dwt(x,wave,mode='symmetric')
A1=pywt.idwt(a1,np.zeros_like(d1),wave,mode='symmetric')[:len(x)]
D1=pywt.idwt(np.zeros_like(a1),d1,wave,mode='symmetric')[:len(x)]
check('E7.10: para gałęzi odtwarza sygnał',np.allclose(A1+D1,x,atol=1e-12))
check('E7.10: symmetric ma zgodną długość współczynników',len(a1)==len(d1)==(len(x)+wave.dec_len-1)//2)
fig,ax=plt.subplots(2,2,figsize=(10,7),layout='constrained')
for filt,label in [(wave.dec_lo,'h analizy LP'),(wave.dec_hi,'g analizy HP')]:ax[0,0].plot(filt,'o-',label=label)
ax[0,0].set(xlabel='Indeks filtru',ylabel='Współczynnik [1]',title='a) Para db4');ax[0,0].legend(fontsize=7)
f,H=signal.freqz(wave.dec_lo,worN=2048,fs=200);_,G=signal.freqz(wave.dec_hi,worN=2048,fs=200)
ax[0,1].plot(f,abs(H),label='LP');ax[0,1].plot(f,abs(G),label='HP');ax[0,1].set(xlabel='Częstotliwość [Hz]',ylabel='Moduł [1]',title='b) Rzeczywiste pasma przejściowe');ax[0,1].legend()
ax[1,0].plot(a1,label='a1');ax[1,0].plot(d1,label='d1');ax[1,0].set(xlabel='Indeks współczynnika',ylabel='Wartość [j.u.]',title=f'c) Po decymacji: {len(a1)}+{len(d1)} współczynników');ax[1,0].legend()
ax[1,1].plot(t,x,label='x',lw=1.5);ax[1,1].plot(t,A1,'--',label='A1');ax[1,1].plot(t,D1,':',label='D1');ax[1,1].set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='d) Rekonstrukcje mają pierwotną oś');ax[1,1].legend()
RESULTS['E7.10']={'N':len(x),'length_a1':len(a1),'length_d1':len(d1),'filter_length':wave.dec_len,'max_reconstruction_error':float(np.max(abs(A1+D1-x)))}
save_figure(fig,FIG,'fig_07_10_dwt_filter_bank')


OK: E7.10: para gałęzi odtwarza sygnał
OK: E7.10: symmetric ma zgodną długość współczynników
Figura: figures/generated/fig_07_10_dwt_filter_bank.png (+ SVG)


## E7.11 — współczynniki, rekonstrukcje i energia; rys. 7.11

Model EKG M6.6:fs200 Hz,T8 s, rytm60/min,R od0.75 s. Gaussy P/Q/R/S/T:
przesunięcia(-0.20,-0.035,0,0.028,0.24)s; amplitudy(0.11,-0.16,1,-0.24,0.28);
sigma(0.035,0.012,0.014,0.014,0.075)s. Dryft0.35Hz a0.22(phi0.5),
HF58Hz a0.075(phi0.2), biały szumSD0.025 seed6606.
db4,J4,symmetric. Energia jest sumą kwadratów próbek, nie całką fizyczną.
Dokładny bilans energii zawiera iloczyny skalarne osobnych rekonstrukcji.

![Rys. 7.11](../figures/generated/fig_07_11_dwt_reconstruction.png)


In [12]:
te,clean,observed,drift,hf,beats=make_synthetic_ecg()
coeff,parts,full=components(observed);labels=list(parts)
sum_parts=sum(parts.values());energy=float(np.sum(observed**2));energy_sum=float(sum(np.sum(v*v) for v in parts.values()))
cross=float(2*sum(np.dot(parts[labels[i]],parts[labels[j]]) for i in range(len(labels)) for j in range(i+1,len(labels))))
check('E7.11: wszystkie gałęzie i ich osobne sumy odtwarzają wejście',np.allclose(full,observed,atol=1e-12) and np.allclose(sum_parts,observed,atol=1e-12))
check('E7.11: bilans energii z wyrazami krzyżowymi',np.isclose(energy,energy_sum+cross,atol=1e-10))
rows=[]
for label,c in zip(labels,coeff):
    j=4 if label=='A4' else int(label[1:]);lo=0 if label=='A4' else 200/2**(j+1);hi=200/2**(j+1) if label=='A4' else 200/2**j
    rows.append({'component':label,'coefficients':len(c),'approx_band_Hz':[lo,hi],
                 'sample_energy':float(np.sum(parts[label]**2)),'RMS':float(np.sqrt(np.mean(parts[label]**2)))})
fig,ax=plt.subplots(6,1,figsize=(9.4,9.3),sharex=True,layout='constrained')
ax[0].plot(te,observed,lw=.6,label='Obserwowany');ax[0].plot(te,clean,lw=.9,label='Czysty');ax[0].legend(ncol=2,fontsize=7);ax[0].set_title('EKG kontrolny: DWT rozdziela podpasma, nie źródła')
for a,label in zip(ax[1:],labels):
    a.plot(te,parts[label],lw=.8);a.set_ylabel(label+' [j.u.]')
ax[0].set_ylabel('x [j.u.]');ax[-1].set_xlabel('Czas [s]')
RESULTS['E7.11']={'components':rows,'N':len(observed),'coefficient_count':sum(len(c) for c in coeff),
                  'energy_input':energy,'sum_component_energies':energy_sum,'cross_terms':cross,
                  'max_sum_error':float(np.max(abs(sum_parts-observed)))}
save_figure(fig,FIG,'fig_07_11_dwt_reconstruction')


OK: E7.11: wszystkie gałęzie i ich osobne sumy odtwarzają wejście
OK: E7.11: bilans energii z wyrazami krzyżowymi
Figura: figures/generated/fig_07_11_dwt_reconstruction.png (+ SVG)


## E7.12 — przesunięcie i wybór falki; rys. 7.12

Ten sam EKG M6.6. Przesunięcie1 próbka=5 ms, brzeg uzupełniony pierwszą wartością.
db4 i sym4,J4,symmetric; oba filtry8 współczynników,4 momenty zanikające.
Porównanie D2 po cofnięciu przesunięcia, wyłącznie0.5–7.5 s.
Dokładna pełna rekonstrukcja współistnieje z różnicami pojedynczych komponentów.


![Rys. 7.12](../figures/generated/fig_07_12_dwt_shift_wavelets.png)


In [13]:
shifted=np.r_[observed[0],observed[:-1]]
c0,p0,r0=components(observed);cs,ps,rs=components(shifted)
aligned=np.r_[ps['D2'][1:],ps['D2'][-1]];inner=(te>=.5)&(te<7.5)
rel=float(np.linalg.norm(aligned[inner]-p0['D2'][inner])/np.linalg.norm(p0['D2'][inner]))
check('E7.12: przesunięcie nie niszczy pełnej rekonstrukcji',np.allclose(rs,shifted,atol=1e-12))
check('E7.12: wyrównanie nie czyni D2 identycznym',rel>.05)
fig,ax=plt.subplots(2,2,figsize=(10,7.3),layout='constrained');rows=[]
for name in ['db4','sym4']:
    wave=pywt.Wavelet(name);phi,psi,grid=wave.wavefun(level=8)
    ax[0,0].plot(grid,psi,label=name)
    cc,pp,rr=components(observed,wavelet=name)
    ax[0,1].plot(te,pp['D2'],label=name)
    rows.append({'wavelet':name,'max_reconstruction_error':float(np.max(abs(rr-observed))),
                 'D2_energy':float(np.sum(pp['D2']**2)),'D3_energy':float(np.sum(pp['D3']**2))})
    check('E7.12: rekonstrukcja '+name,np.max(abs(rr-observed))<1e-10)
ax[0,0].set(xlabel='Czas znormalizowany [1]',ylabel='Wartość ψ [1]',title='a) Cztery momenty, różne kształty');ax[0,0].legend()
ax[0,1].set(xlim=(2.5,3.1),xlabel='Czas [s]',ylabel='D2 [j.u.]',title='b) Ten sam poziom nie oznacza tej samej cechy');ax[0,1].legend()
idx=list(p0).index('D2')
ax[1,0].plot(c0[idx],label='d2 oryginalne');ax[1,0].plot(cs[idx],'--',label='d2 po przesunięciu');ax[1,0].set(xlabel='Indeks współczynnika',ylabel='Wartość [j.u.]',title='c) Zmiana fazy decymacji');ax[1,0].legend(fontsize=7)
ax[1,1].plot(te,p0['D2'],label='D2 oryginalne');ax[1,1].plot(te,aligned,'--',label='D2 po wyrównaniu');ax[1,1].set(xlim=(2.5,3.1),xlabel='Czas [s]',ylabel='D2 [j.u.]',title=f'd) Względna różnica wnętrza: {rel:.3f}');ax[1,1].legend(fontsize=7)
RESULTS['E7.12']={'aligned_D2_relative_L2_difference':rel,'shift_s':.005,'wavelets':rows}
save_figure(fig,FIG,'fig_07_12_dwt_shift_wavelets')


OK: E7.12: przesunięcie nie niszczy pełnej rekonstrukcji
OK: E7.12: wyrównanie nie czyni D2 identycznym
OK: E7.12: rekonstrukcja db4
OK: E7.12: rekonstrukcja sym4
Figura: figures/generated/fig_07_12_dwt_shift_wavelets.png (+ SVG)


## E7.13 — progowanie ma koszt morfologiczny; rys. 7.13

Model M6.7: fs250 Hz,10 s,R co1 s od0.8 s; gaussy P/Q/R/S/T:
przesunięcia(-0.19,-0.030,0,0.026,0.24)s,amplitudy(0.10,-0.16,1,-0.24,0.27),
sigma(0.035,0.010,0.013,0.013,0.070)s. SzumSD0.080 seed6701,
65 Hz a0.045(phi0.2), artefakty+0.34G(3.35,0.018)−0.28G(7.15,0.022).
db4,J4,symmetric; sigma_hat=median|d1|/0.6745,lambda=sigma_hat*sqrt(2lnN).
Hard/soft na wszystkich detalach; aproksymacja bez zmian.
R: znane okna±0.07 s, amplitudy lokalnych maksimów bez korekty linii bazowej;
czasy względem maksimów czystego modelu, nie nominalnego centrum Gaussa.

![Rys. 7.13](../figures/generated/fig_07_13_thresholding.png)


In [14]:
td,ref,noisy,total,stationary,artifacts,rtimes=make_ecg_wavelet_case();fsd=250.
base_metrics=metrics(ref,noisy,fsd,rtimes);denoised={};method_rows=[{'method':'Wejście',**base_metrics}]
for rule in ['hard','soft']:
    y,coeff,kept,sigma,lam=denoise(noisy,rule=rule)
    denoised[rule]=y;method_rows.append({'method':rule,**metrics(ref,y,fsd,rtimes)})
    check('E7.13: aproksymacja niezmieniona '+rule,np.array_equal(coeff[0],kept[0]))
    check('E7.13: zachowana długość i skończoność '+rule,len(y)==len(noisy) and np.all(np.isfinite(y)))
check('E7.13: soft zmniejsza wszystkie zachowane detale',all(np.all(abs(b)<=abs(a)+1e-14) for a,b in zip(coeff[1:],kept[1:])))
fig,ax=plt.subplots(2,2,figsize=(10.2,7.2),layout='constrained')
d1=coeff[-1];ax[0,0].hist(d1,bins=65,color='#28738A',alpha=.8);ax[0,0].axvline(sigma,color='#B25324',ls='--',label='σ̂');ax[0,0].axvline(-sigma,color='#B25324',ls='--')
ax[0,0].set(xlabel='Współczynnik d1 [j.u.]',ylabel='Liczba',title='a) Model szumu w najdrobniejszym detalu');ax[0,0].legend()
xx=np.linspace(-.8,.8,1000)
for rule in ['hard','soft']:ax[0,1].plot(xx,pywt.threshold(xx,lam,mode=rule),label=rule)
ax[0,1].plot(xx,xx,':',color='black',lw=.8)
ax[0,1].set(xlabel='Współczynnik przed [j.u.]',ylabel='Współczynnik po [j.u.]',title=f'b) Wspólny próg λ={lam:.3f}');ax[0,1].legend()
ax[1,0].plot(td,noisy,color='0.75',lw=.7,label='Wejście');ax[1,0].plot(td,ref,'k',lw=1.5,label='Odniesienie')
for rule,y in denoised.items():ax[1,0].plot(td,y,label=rule)
ax[1,0].set(xlim=(2.7,3.85),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='c) R i artefakt nie są tą samą strukturą');ax[1,0].legend(ncol=2,fontsize=7)
zero_rows=[]
for alpha in [.5,1.,1.5]:
    threshold=alpha*lam
    zeros=[float(np.mean(abs(c)<threshold)) for c in coeff[1:]]
    ax[1,1].plot([4,3,2,1],np.array(zeros)*100,'o-',label=f'α={alpha:g}')
    zero_rows.append({'alpha':alpha,'fraction_below_threshold_D4_D1':zeros})
ax[1,1].set(xlabel='Poziom detalu j',ylabel='Współczynniki poniżej progu [%]',title='d) Ten sam próg oddziałuje na różne poziomy');ax[1,1].legend()
RESULTS['E7.13']={'sigma_hat':sigma,'white_noise_SD':.08,'stationary_added_SD':float(np.std(stationary)),
    'threshold_alpha1':lam,'methods':method_rows,'fractions':zero_rows}
save_figure(fig,FIG,'fig_07_13_thresholding')


OK: E7.13: aproksymacja niezmieniona hard
OK: E7.13: zachowana długość i skończoność hard
OK: E7.13: aproksymacja niezmieniona soft
OK: E7.13: zachowana długość i skończoność soft
OK: E7.13: soft zmniejsza wszystkie zachowane detale
Figura: figures/generated/fig_07_13_thresholding.png (+ SVG)


## E7.14 — błąd globalny, cecha chroniona i nierozdzielny artefakt; rys. 7.14

To samo wejście M6.7. Soft,J3/4/5,alpha0.5/0.75/1/1.25/1.5.
Przykładowa tolerancja max|błądR|<=0.05 j.u.; brak wariantu nie jest akceptacją
minimumRMSE. BP:Butterworth N4 dla przekształcenia pasmowego (rzeczywisty rząd8),
0.5–40 Hz,SOS,przód–tył,odd,padlen27. Brzegi: pierwsze/ostatnie0.5 s.
Próba niepowodzenia M6.7.6:0.75G(5.35,0.014)−0.25G(5.378,0.013),
szumSD0.045,seed6706. Rosnący próg porównywany równocześnie z błędemR.

![Rys. 7.14](../figures/generated/fig_07_14_denoising_audit.png)


In [15]:
search=[]
for J in [3,4,5]:
    for alpha in [.5,.75,1.,1.25,1.5]:
        y,*_=denoise(noisy,level=J,alpha=alpha)
        search.append({'J':J,'alpha':alpha,**metrics(ref,y,fsd,rtimes)})
best=min(search,key=lambda r:r['RMSE']);feasible=[r for r in search if r['R_amp_max_abs']<=.05]
ybest,*_=denoise(noisy,level=best['J'],alpha=best['alpha'])
sos=signal.butter(4,[.5,40],btype='bandpass',fs=250,output='sos')
yfilter=signal.sosfiltfilt(sos,noisy,padtype='odd',padlen=27)
comparison=method_rows+[{'method':f"Soft min RMSE J{best['J']} α{best['alpha']:g}",**metrics(ref,ybest,fsd,rtimes)},
                        {'method':'BP 0,5–40 Hz',**metrics(ref,yfilter,fsd,rtimes)}]
fig,ax=plt.subplots(2,2,figsize=(10.3,7.5),layout='constrained')
for J in [3,4,5]:
    group=[r for r in search if r['J']==J]
    ax[0,0].plot([r['alpha'] for r in group],[r['RMSE'] for r in group],'o-',label=f'J={J}')
    ax[0,1].plot([r['alpha'] for r in group],[r['R_amp_max_abs'] for r in group],'o-',label=f'J={J}')
ax[0,0].set(xlabel='Mnożnik α [1]',ylabel='RMSE [j.u.]',title='a) Minimum błędu globalnego')
ax[0,1].axhline(.05,color='black',ls=':',label='Tolerancja0,05')
ax[0,1].set(xlabel='Mnożnik α [1]',ylabel='Maks. |błąd amplitudy R| [j.u.]',title='b) Ta sama siatka, inny warunek oceny')
for a in ax[0]:a.legend(fontsize=7)
same_art=.75*gaussian_envelope(td,5.35,.014)-.25*gaussian_envelope(td,5.378,.013)
failure=ref+.045*np.random.default_rng(6706).normal(size=len(td))+same_art
failure_rows=[]
ax[1,0].plot(td,ref,'k',lw=1.5,label='Czysty');ax[1,0].plot(td,failure,color='0.75',lw=.7,label='Z artefaktem')
for alpha in [.5,1.,1.5,2.]:
    y,*_=denoise(failure,alpha=alpha);m=metrics(ref,y,fsd,rtimes);roi=abs(td-5.35)<.08
    failure_rows.append({'alpha':alpha,'artifact_window_RMSE':rmse(y[roi],ref[roi]),**m})
    ax[1,0].plot(td,y,label=f'α={alpha:g}')
ax[1,0].set(xlim=(4.7,5.9),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='c) Artefakt podobny do QRS przetrwa próg');ax[1,0].legend(ncol=3,fontsize=7)
ax[1,1].plot([r['artifact_window_RMSE'] for r in failure_rows],[r['R_amp_max_abs'] for r in failure_rows],'o-')
for r in failure_rows:ax[1,1].annotate(f"α={r['alpha']:g}",(r['artifact_window_RMSE'],r['R_amp_max_abs']),xytext=(4,4),textcoords='offset points',fontsize=7)
ax[1,1].set(xlabel='RMSE okna artefaktu [j.u.]',ylabel='Maks. |błąd R| [j.u.]',title='d) Tłumienie artefaktu ma koszt użytecznej cechy')
check('E7.14: cała siatka parametrów została oceniona',len(search)==15)
check('E7.14: wynik minimumRMSE istnieje i jest skończony',np.isfinite(best['RMSE']))
RESULTS['E7.14']={'parameter_search':search,'minimum_RMSE':best,'amplitude_tolerance':.05,
    'feasible_count':len(feasible),'comparison':comparison,'failure':failure_rows,
    'BP':{'prototype_order':4,'actual_order':8,'sos_sections':len(sos),'band_Hz':[.5,40],'padtype':'odd','padlen':27,'direction':'forward-backward'}}
table='| Wariant | RMSE [j.u.] | SNR [dB] | Maks. błąd R [j.u.] | Maks. błąd czasu R [ms] | RMSE brzegów [j.u.] |\n|---|---:|---:|---:|---:|---:|\n'
for r in comparison:
    table+='| '+r['method']+' | '+' | '.join(f"{r[k]:.3f}".replace('.',',') for k in ['RMSE','SNR_dB','R_amp_max_abs','R_time_max_abs_ms','edge_RMSE'])+' |\n'
(ROOT/'data/ch07_denoising_table.md').write_text(table, encoding='utf-8', newline='\n')
save_figure(fig,FIG,'fig_07_14_denoising_audit')


OK: E7.14: cała siatka parametrów została oceniona
OK: E7.14: wynik minimumRMSE istnieje i jest skończony
Figura: figures/generated/fig_07_14_denoising_audit.png (+ SVG)


## E7.15 — gdy słownik reprezentacji nie identyfikuje źródeł; rys. 7.15

Model końcowy M6.8,fs200 Hz,8 s,seed6803. Dwie nieregularne składowe,
zmienne amplitudy, przecięcia i skoki praw częstotliwości, stan przejściowy.
STFT:Hann,L96/H12/NFFT512 orazL256/H32/NFFT1024.
CWT:cmor0.8-1.0 i cmor2.5-1.0,180 częstotliwości3–40 Hz.
Każda mapa normalizowana do własnego maksimum; służy porównaniu struktur.
Porównanie dominujących grzbietów na wspólnych środkach pełnych ramek STFT,
wewnątrz zakresu czasów obu wariantów i w3–5.5 s.
DWTdb4,J5,symmetric: rozkład energii detali nie identyfikuje osobnych źródeł.

![Rys. 7.15](../figures/generated/fig_07_15_representation_limit.png)


In [16]:
t,x,c1,c2,trans,f1,f2,a1,a2=make_mismatched_case();fig,ax=plt.subplots(2,2,figsize=(10.8,8),layout='constrained')
stages=[];cstages=[]
for a,(L,H,NF) in zip(ax[0],[(96,12,512),(256,32,1024)]):
    r=stft_map(x,length=L,hop=H,nfft=NF);band=(r['f']>=3)&(r['f']<=40)
    ridge=r['f'][band][np.argmax(r['power'][band],axis=0)];stages.append((r['time'],ridge))
    tfplot(a,r['time'],r['f'],r['power'],f'STFT L={L}, H={H}',limits=(-55,0),ylim=(3,40))
    a.plot(t,f1,'w--',lw=.7);a.plot(t,f2,'w:',lw=.7);a.plot(r['time'],ridge,color='#F7C77A',lw=.8)
for a,wave in zip(ax[1],['cmor0.8-1.0','cmor2.5-1.0']):
    r=cwt_map(x,frequencies=np.geomspace(3,40,180),wavelet=wave)
    ridge=r['f'][np.argmax(r['power'],axis=0)];cstages.append(ridge)
    tfplot(a,t,r['f'],r['power'],wave,kind='CWT',limits=(-55,0),ylim=(3,40))
    a.plot(t,f1,'w--',lw=.7);a.plot(t,f2,'w:',lw=.7);a.plot(t,ridge,color='#F7C77A',lw=.8)
for a in ax.flat:a.set_xlim(0,8)
ts,rs=stages[0];tl,rl=stages[1];common=(ts>=max(.8,tl[0]))&(ts<=min(7.2,tl[-1]))
difference=abs(rs[common]-np.interp(ts[common],tl,rl));region=(ts[common]>=3)&(ts[common]<=5.5)
cw_difference=abs(cstages[0]-cstages[1]);cregion=(t>=3)&(t<=5.5)
co,pa,re=components(x,level=5);energies={name:float(np.sum(v*v)) for name,v in pa.items()}
check('E7.15: pełna DWT nadal odtwarza złożone wejście',np.allclose(re,x,atol=1e-12))
check('E7.15: parametry zmieniają dominujący grzbiet',np.max(difference[region])>1 and np.max(cw_difference[cregion])>1)
RESULTS['E7.15']={'STFT_ridge_difference_p90_Hz_in3_5_5':float(np.percentile(difference[region],90)),
    'STFT_ridge_max_difference_Hz_in3_5_5':float(np.max(difference[region])),
    'CWT_ridge_difference_p90_Hz_in3_5_5':float(np.percentile(cw_difference[cregion],90)),
    'DWT_sample_energies':energies}
save_figure(fig,FIG,'fig_07_15_representation_limit')


OK: E7.15: pełna DWT nadal odtwarza złożone wejście
OK: E7.15: parametry zmieniają dominujący grzbiet
Figura: figures/generated/fig_07_15_representation_limit.png (+ SVG)


## Raport wykonania

Pełne wyniki, paszporty i wersje bibliotek są zapisane obok rozdziału.
Kontrole weryfikują obliczenia i właściwości modeli, nie rozpoznania kliniczne.


In [17]:
for i in range(1,16):
    for ext in ['png','svg']:
        check(f'Figura7.{i} {ext}',len(list(FIG.glob(f'fig_07_{i:02d}_*.{ext}')))==1)
RESULTS['audit']={'passed_checks':len(CHECKS),'checks':CHECKS,'experiments':15,'figures':15,
                  'external_data':False,'synthetic_data':True}
(ROOT/'data/ch07_results.json').write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2,allow_nan=False)+'\n', encoding='utf-8', newline='\n')
(ROOT/'data/ch07_parameter_passports.json').write_text(json.dumps(PASSPORTS,ensure_ascii=False,indent=2,allow_nan=False)+'\n', encoding='utf-8', newline='\n')
(ROOT/'chapter_notes/ch07_environment.json').write_text(json.dumps(ENV,ensure_ascii=False,indent=2)+'\n', encoding='utf-8', newline='\n')
print('Zakończono. Liczba kontroli:',len(CHECKS))


OK: Figura7.1 png
OK: Figura7.1 svg
OK: Figura7.2 png
OK: Figura7.2 svg
OK: Figura7.3 png
OK: Figura7.3 svg
OK: Figura7.4 png
OK: Figura7.4 svg
OK: Figura7.5 png
OK: Figura7.5 svg
OK: Figura7.6 png
OK: Figura7.6 svg
OK: Figura7.7 png
OK: Figura7.7 svg
OK: Figura7.8 png
OK: Figura7.8 svg
OK: Figura7.9 png
OK: Figura7.9 svg
OK: Figura7.10 png
OK: Figura7.10 svg
OK: Figura7.11 png
OK: Figura7.11 svg
OK: Figura7.12 png
OK: Figura7.12 svg
OK: Figura7.13 png
OK: Figura7.13 svg
OK: Figura7.14 png
OK: Figura7.14 svg
OK: Figura7.15 png
OK: Figura7.15 svg
Zakończono. Liczba kontroli: 78
